# GeoStat_py · Workbench principal autosuficiente (Colab)

Notebook principal para uso real en Colab, **sin depender de ejecutar otro notebook**.

Flujo único:
1) bootstrap del entorno,
2) creación de `service`,
3) upload CSV local,
4) carga + autodetección,
5) configuración X/Y/Z/target,
6) EDA inline,
7) variografía inline.


In [ ]:
# 0) Bootstrap autosuficiente (sesión fresca de Colab)
from pathlib import Path
import subprocess
import sys

# Configuración base (editable)
REPO_URL = "https://github.com/joelmanrique91-lgtm/GeoStat_py.git"
BRANCH = "main"
BASE_DIR = "/content"
REPO_DIR_NAME = "GeoStat_py"
REPO_DIR = Path(BASE_DIR) / REPO_DIR_NAME

# 0.1 Asegurar repo local para poder importar bootstrap.py
if not REPO_DIR.exists():
    print(f"Repo no encontrado en {REPO_DIR}. Clonando...")
    clone_cmd = ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)]
    clone_result = subprocess.run(clone_cmd, text=True, capture_output=True, check=False)
    print(clone_result.stdout)
    if clone_result.returncode != 0:
        print(clone_result.stderr)
        raise RuntimeError(f"No se pudo clonar el repo (code={clone_result.returncode}).")

# 0.2 Importar helpers del bootstrap de esta capa colab/
bootstrap_module_path = str(REPO_DIR / "colab")
if bootstrap_module_path in sys.path:
    sys.path.remove(bootstrap_module_path)
sys.path.insert(0, bootstrap_module_path)

from bootstrap import (
    clone_or_update_repo,
    configure_sys_path,
    create_service,
    install_requirements,
    validate_imports,
)

# 0.3 Actualizar repo, instalar deps mínimas, configurar path del proyecto
repo_root = clone_or_update_repo(REPO_URL, REPO_DIR, BRANCH)
install_requirements(Path(repo_root) / "colab" / "requirements_colab.txt")
configure_sys_path(repo_root)

# 0.4 Validar imports del motor ANTES de trabajar
imports_to_check = [
    "app.services.geostat_service",
    "app.services.visualization_service",
    "app.services.variography_application_service",
    "app.models.dataset_model",
]
results = validate_imports(imports_to_check)
all_ok = True
for item in results:
    status = "OK" if item.ok else "ERROR"
    print(f"[{status}] {item.module_name}")
    if item.ok:
        print("   origin:", item.origin)
    else:
        all_ok = False
        print("   error:", item.error)
if not all_ok:
    raise RuntimeError("Falló la validación de imports del motor.")

# 0.5 Crear servicio listo para notebook
service = create_service()
print("Service listo:", type(service).__name__)


In [ ]:
# 1) Imports de trabajo analítico (después del bootstrap)
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

UPLOAD_DIR = Path("/content/geostat_uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("UPLOAD_DIR:", UPLOAD_DIR)


In [ ]:
# 2) Parámetros editables (columnas + variografía)
# Si dejas vacío, se usa autodetección del servicio.
X_COLUMN_OVERRIDE = ""
Y_COLUMN_OVERRIDE = ""
Z_COLUMN_OVERRIDE = ""   # si queda vacío, se crea z sintética=0.0
TARGET_COLUMN_OVERRIDE = ""
DOMAIN_COLUMN_OVERRIDE = ""  # opcional
HOLE_ID_COLUMN_OVERRIDE = ""  # opcional

VARIO_LAG_DISTANCE = 10.0
VARIO_N_LAGS = 12
VARIO_LAG_TOLERANCE = 5.0
VARIO_MAX_DISTANCE = 120.0
VARIO_AZIMUTH = 0.0
VARIO_DIP = 0.0
VARIO_ANG_TOL_H = 90.0
VARIO_ANG_TOL_V = 90.0
VARIO_BAND_WIDTH = 0.0
VARIO_BAND_HEIGHT = 0.0
VARIO_ESTIMATOR = "classical"


In [ ]:
# 3) Upload CSV local desde memoria del usuario (opcional pero recomendado)
from google.colab import files

csv_path = None
uploaded = files.upload()
if uploaded:
    uploaded_name = list(uploaded.keys())[0]
    uploaded_bytes = uploaded[uploaded_name]
    csv_path = UPLOAD_DIR / uploaded_name
    csv_path.write_bytes(uploaded_bytes)
    print("CSV subido:", uploaded_name)
    print("Ruta estable:", csv_path)
else:
    print("No se subió CSV en esta ejecución. Puedes volver a correr esta celda cuando quieras.")


In [ ]:
# 4) Cargar dataset + mostrar columnas y autodetección
columns = []
autodetected = {}

dataset_loaded = False
if csv_path is None:
    print("Dataset no cargado: falta CSV. Ejecuta la celda de upload.")
else:
    load_result = service.load_csv(str(csv_path))
    print("load_csv.success:", load_result.success)
    print("load_csv.message:", load_result.message)
    print("load_csv.details:", load_result.details)

    dataset_loaded = bool(load_result.success)
    if dataset_loaded:
        columns = service.get_available_columns()
        autodetected = service.get_autodetected_columns()
        print("\nColumnas disponibles:")
        print(columns)
        print("\nAutodetección del servicio:")
        print(json.dumps(autodetected, ensure_ascii=False, indent=2))


In [ ]:
# 5) Configurar X/Y/Z/target con APIs reales del servicio
config_applied = False
resolved_target_col = ""

if not dataset_loaded:
    print("Configuración omitida: primero carga un dataset.")
else:
    x_col = (X_COLUMN_OVERRIDE or autodetected.get("x") or "").strip()
    y_col = (Y_COLUMN_OVERRIDE or autodetected.get("y") or "").strip()
    z_col = (Z_COLUMN_OVERRIDE or autodetected.get("z") or "").strip()
    target_col = (TARGET_COLUMN_OVERRIDE or autodetected.get("target") or "").strip()
    domain_col = (DOMAIN_COLUMN_OVERRIDE or autodetected.get("domain") or "").strip()
    hole_id_col = (HOLE_ID_COLUMN_OVERRIDE or autodetected.get("hole_id") or "").strip()

    print("Selección inicial:")
    print({"x": x_col, "y": y_col, "z": z_col, "target": target_col, "domain": domain_col, "hole_id": hole_id_col})

    if not x_col or not y_col or not target_col:
        print("No se pudo resolver X/Y/target. Define overrides y vuelve a ejecutar esta celda.")
    else:
        if not z_col:
            synthetic_z_col = "__z_colab__"
            if service.current_dataset is None:
                raise RuntimeError("No hay dataset cargado para crear Z sintética.")
            df_tmp = service.current_dataset.dataframe.copy()
            df_tmp[synthetic_z_col] = 0.0
            synthetic_csv_path = UPLOAD_DIR / f"{Path(csv_path).stem}__with_z.csv"
            df_tmp.to_csv(synthetic_csv_path, index=False)
            print(f"Z vacía: se creó `{synthetic_z_col}` y CSV temporal {synthetic_csv_path}")

            reload_result = service.load_csv(str(synthetic_csv_path))
            print("reload.success:", reload_result.success)
            if not reload_result.success:
                raise RuntimeError(f"No se pudo recargar dataset con Z sintética: {reload_result.message}")

            z_col = synthetic_z_col

        cfg_result = service.set_variable_config(
            x_column=x_col,
            y_column=y_col,
            z_column=z_col,
            target_column=target_col,
            hole_id_column=hole_id_col or None,
            domain_column=domain_col or None,
        )
        print("\nset_variable_config.success:", cfg_result.success)
        print("set_variable_config.message:", cfg_result.message)
        print("set_variable_config.eda_summary:", cfg_result.eda_summary)

        config_applied = bool(cfg_result.success)
        resolved_target_col = target_col


In [ ]:
# 6) EDA inline (si hay configuración aplicada)
if not config_applied:
    print("EDA omitido: primero aplica configuración X/Y/Z/target.")
else:
    summary_text = service.build_eda_summary()
    stats_rows = service.get_target_statistics_table()
    print("Resumen EDA:", summary_text)

    if stats_rows:
        display(pd.DataFrame(stats_rows, columns=["metric", "value"]))
    else:
        print("No hay tabla de estadísticas disponible.")

    try:
        univariate = service.prepare_univariate_data()
        target_values = univariate.get("target_values", [])
        prob_x = univariate.get("probplot_x", [])
        prob_y = univariate.get("probplot_y", [])

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        if target_values:
            axes[0].hist(target_values, bins=30, color="#1f77b4", alpha=0.85)
            axes[0].set_title("Histograma target")
            axes[1].boxplot(target_values, vert=True)
            axes[1].set_title("Boxplot target")
        else:
            axes[0].text(0.5, 0.5, "Sin datos target", ha="center", va="center")
            axes[1].text(0.5, 0.5, "Sin datos target", ha="center", va="center")

        if prob_x and prob_y:
            axes[2].scatter(prob_x, prob_y, s=10, alpha=0.8)
            axes[2].set_title("Probability plot")
            axes[2].set_xlabel("Cuantiles teóricos")
            axes[2].set_ylabel("Valores ordenados")
        else:
            axes[2].text(0.5, 0.5, "Probability no disponible", ha="center", va="center")

        plt.tight_layout()
        plt.show()
    except Exception as exc:  # noqa: BLE001
        print(f"EDA no ejecutable con este dataset/configuración: {exc}")


In [ ]:
# 7) Variografía inline (si hay configuración aplicada)
if not config_applied:
    print("Variografía omitida: primero aplica configuración X/Y/Z/target.")
else:
    variography_params = {
        "target_col": resolved_target_col,
        "lag_distance": float(VARIO_LAG_DISTANCE),
        "n_lags": int(VARIO_N_LAGS),
        "lag_tolerance": float(VARIO_LAG_TOLERANCE),
        "max_distance": float(VARIO_MAX_DISTANCE),
        "azimuth": float(VARIO_AZIMUTH),
        "dip": float(VARIO_DIP),
        "ang_tol_h": float(VARIO_ANG_TOL_H),
        "ang_tol_v": float(VARIO_ANG_TOL_V),
        "band_width": float(VARIO_BAND_WIDTH),
        "band_height": float(VARIO_BAND_HEIGHT),
        "estimator": str(VARIO_ESTIMATOR),
    }

    response = service.compute_experimental_variography(variography_params)
    print("ok:", response.ok)
    print("message:", response.message)
    print("warnings:", [w.code for w in response.warnings])
    print("blockers:", [b.code for b in response.blockers])

    if response.result is None:
        print("No hay resultado numérico de variografía para graficar.")
    else:
        lags = response.result.lag_centers
        gamma = response.result.gamma_values
        pairs = response.result.pair_counts
        display(pd.DataFrame({"lag_center": lags, "gamma": gamma, "npairs": pairs}))

        fig, ax1 = plt.subplots(figsize=(8, 4))
        ax1.plot(lags, gamma, marker="o", color="#2ca02c")
        ax1.set_xlabel("Lag center")
        ax1.set_ylabel("Semivarianza")
        ax1.set_title("Variograma experimental")

        ax2 = ax1.twinx()
        width = max(1.0, (max(lags) / max(1, len(lags))) * 0.6)
        ax2.bar(lags, pairs, alpha=0.18, width=width, color="#ff7f0e")
        ax2.set_ylabel("N pares")

        plt.tight_layout()
        plt.show()


## Estado final esperado
- Este notebook funciona desde una sesión fresca de Colab.
- No depende de `00_bootstrap.ipynb`.
- Si no subes CSV aún, no rompe el flujo; puedes subirlo y continuar.
- Si no hay Z, crea Z sintética de forma segura para mantener compatibilidad con el servicio actual.
